# Redrob Ranker — Full Pipeline Notebook (v2)

**Two phases:**
- **PRECOMPUTE** (run once, ~20–40 min on GPU / 2–4 hrs CPU): embeds 300K job texts → `.npy` files on disk
- **RANKING** (< 5 min, no network, CPU-only sandbox): loads `.npy`, matrix-multiplies, applies all multipliers, outputs `top100_ranked.csv`

> ⚠️ **Primary signal: `candidate_jobs.description` — NOT the skills table** (skills = honeypot trap per JD)
>
> Ranking uses geometric mean of must-haves (M1–M4) so a near-zero on any dimension kills the candidate.

**v2 fixes applied:**
- `description_confidence()` NaN-safe via `safe_text()`
- `candidate_base.summary` / `headline` merged in and used for sparse job descriptions
- Fixed `REFERENCE_DATE` (reproducible, env-var overridable)
- Trap multiplier requires explicit column — no boolean accidents
- `df_cand` seeded from all `survivor_ids` (not just those with job rows)
- `duration_months` always recomputed after date cleanup
- Best-job selection uses geometric mean (not single-M max)


In [8]:
import os, time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import gmean as scipy_gmean
import torch

# ── Paths (adjust if your layout differs) ─────────────────────────────────
DATA_DIR   = Path(r'E:\Coding\Projects\redrob-ranker\dataset\artifacts')
EMBED_DIR  = Path('artifacts'); EMBED_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path('output');    OUTPUT_DIR.mkdir(exist_ok=True)

# ── GPU detection ──────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    BATCH_SIZE = 512
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("Estimated encode time: ~20–40 min")
else:
    BATCH_SIZE = 128
    print("No GPU — using CPU (expect 2–4 hours for encoding)")

# FIX: Fixed reference date for reproducibility.
# Override via environment variable if needed: REFERENCE_DATE=2026-06-28 jupyter notebook
REFERENCE_DATE = pd.Timestamp(os.getenv("REFERENCE_DATE", "2026-06-25"))
QUERY_KEYS = ['M1','M2','M3','M4','N1','N2','N3','N4','IR','CV','HANDS_ON']
GMEAN_FLOOR = 0.05

print(f"\nReference date : {REFERENCE_DATE.date()}  (set REFERENCE_DATE env var to override)")
print(f"Batch size     : {BATCH_SIZE}")


No GPU — using CPU (expect 2–4 hours for encoding)

Reference date : 2026-06-25  (set REFERENCE_DATE env var to override)
Batch size     : 128


## Part 1 — Data Loading

In [9]:
DATA_DIR
import os
os.getcwd()

(DATA_DIR / 'candidate_base.parquet').exists()

True

In [10]:
df_base      = pd.read_parquet(DATA_DIR / 'candidate_base.parquet')
df_jobs_raw  = pd.read_parquet(DATA_DIR / 'candidate_jobs.parquet')
df_avail     = pd.read_parquet(DATA_DIR / 'availability_scores.parquet')
df_survivors = pd.read_parquet(DATA_DIR / 'survivors.parquet')

# ── Inspect columns — verify these match your parquets before proceeding ──
print("=== candidate_jobs columns ===")
print(df_jobs_raw.dtypes)
print()
print("=== candidate_base columns ===")
print(df_base.dtypes)

# ── Filter job rows to survivors only ─────────────────────────────────────
survivor_ids = set(df_survivors['candidate_id'])
df_jobs = df_jobs_raw[df_jobs_raw['candidate_id'].isin(survivor_ids)].copy()

print(f"\nSurvivors           : {len(survivor_ids):,}")
print(f"Job rows (raw)      : {len(df_jobs_raw):,}")
print(f"Job rows (survivors): {len(df_jobs):,}")
print(f"Avg jobs/candidate  : {len(df_jobs)/len(survivor_ids):.1f}")

# FIX: Track candidates with no job rows — they'll still appear in df_cand
cands_with_jobs = set(df_jobs['candidate_id'].unique())
no_job_cands    = survivor_ids - cands_with_jobs
print(f"Survivors with no job rows: {len(no_job_cands):,}  (will score 0 on all M)")


=== candidate_jobs columns ===
candidate_id         str
job_index          int64
company              str
title                str
start_date           str
end_date             str
duration_months    int64
is_current          bool
industry             str
company_size         str
description          str
dtype: object

=== candidate_base columns ===
candidate_id                str
anonymized_name             str
headline                    str
summary                     str
location                    str
country                     str
years_of_experience     float64
current_title               str
current_company             str
current_company_size        str
current_industry            str
dtype: object

Survivors           : 90,236
Job rows (raw)      : 300,171
Job rows (survivors): 285,346
Avg jobs/candidate  : 3.2
Survivors with no job rows: 0  (will score 0 on all M)


## Part 2 — Precompute Embeddings *(run once — output saved to disk)*

Produces:
- `artifacts/job_embeddings.npy` — shape `(N_survivor_jobs, 768)` ~920 MB
- `artifacts/job_meta.parquet`   — metadata needed at ranking time
- `artifacts/query_embeddings.npy` — shape `(11, 768)`

Once these files exist, skip to **Part 3**.


In [11]:
# ── Must-have queries (M1–M4) ─────────────────────────────────────────────
MUST_HAVE_QUERIES = {
    'M1': (
        'production deployment of embedding-based retrieval or semantic search '
        'systems to real users, handling embedding drift, index refresh, and '
        'retrieval quality regression in production'
    ),
    'M2': (
        'hands-on production experience with vector databases or hybrid search '
        'infrastructure such as FAISS, Pinecone, Weaviate, Qdrant, Milvus, '
        'OpenSearch, or Elasticsearch — operational experience not just tutorials'
    ),
    'M3': (
        'designing and operating ranking evaluation frameworks, NDCG, MRR, MAP, '
        'offline-to-online correlation, A/B testing of ranking or recommendation '
        'systems, understanding how to measure ranking quality rigorously'
    ),
    'M4': (
        'personally implemented and deployed ML or backend systems to production, '
        'wrote production Python code, owned deployed services, hands-on '
        'implementation rather than architecture oversight or team management'
    ),
}

# ── Nice-to-have queries (N1–N4) — bonus only, never penalises absence ────
NICE_TO_HAVE_QUERIES = {
    'N1': (
        'LLM fine-tuning experience — LoRA, QLoRA, PEFT, parameter-efficient '
        'fine-tuning, instruction tuning, RLHF'
    ),
    'N2': (
        'learning-to-rank models, XGBoost LTR, neural ranking, '
        'listwise or pairwise ranking losses, LambdaMART'
    ),
    'N3': (
        'HR technology, recruiting platform, marketplace product, '
        'talent intelligence, job matching, candidate search product'
    ),
    'N4': (
        'distributed systems, large-scale ML inference optimization, '
        'model serving at scale, latency optimization, GPU inference'
    ),
}

# ── Domain + hands-on queries ─────────────────────────────────────────────
DOMAIN_QUERIES = {
    'IR': (
        'information retrieval, NLP, natural language processing, search, ranking, '
        'recommendation systems, text embeddings, semantic search, '
        'retrieval augmented generation, question answering'
    ),
    'CV': (
        'computer vision, image recognition, object detection, speech recognition, '
        'audio processing, robotics, pose estimation, image segmentation, '
        'convolutional neural network'
    ),
}

HANDS_ON_QUERY = (
    'personally implemented and deployed ML or backend systems, '
    'wrote production code, built and shipped features to real users, '
    'hands-on engineering not team management or architecture review'
)

# Ordered query list — index order MUST match QUERY_KEYS
QUERY_TEXTS = [
    MUST_HAVE_QUERIES['M1'], MUST_HAVE_QUERIES['M2'],
    MUST_HAVE_QUERIES['M3'], MUST_HAVE_QUERIES['M4'],
    NICE_TO_HAVE_QUERIES['N1'], NICE_TO_HAVE_QUERIES['N2'],
    NICE_TO_HAVE_QUERIES['N3'], NICE_TO_HAVE_QUERIES['N4'],
    DOMAIN_QUERIES['IR'], DOMAIN_QUERIES['CV'],
    HANDS_ON_QUERY,
]
assert len(QUERY_KEYS) == 11 == len(QUERY_TEXTS), "Query count mismatch"
print("Query keys:", QUERY_KEYS)


Query keys: ['M1', 'M2', 'M3', 'M4', 'N1', 'N2', 'N3', 'N4', 'IR', 'CV', 'HANDS_ON']


In [12]:
# ── NaN-safe text helper (FIX 1) ──────────────────────────────────────────
def safe_text(x):
    """Return empty string for NaN/None; str() everything else."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return str(x).strip()

# ── Tech anchors for confidence scoring ───────────────────────────────────
TECH_ANCHORS = {
    'milvus','pinecone','faiss','elasticsearch','qdrant','weaviate',
    'bert','embedding','vector','retrieval','ndcg','ranking','transformer',
    'fine-tun','pytorch','sklearn','xgboost','rag','llm','semantic',
    'reranking','bm25','dense','sparse','hybrid','sentence-transformer',
}

def description_confidence(text):
    """
    Scalar [0.20, 1.0] penalising vague short descriptions.
    Short-but-technical gets a floor of 0.75.
    FIX 1: uses safe_text() — NaN-safe, no crash.
    """
    text = safe_text(text)         # FIX: was: if not text or not text.strip()
    if not text:
        return 0.20
    wc = len(text.split())
    has_anchor = any(t in text.lower() for t in TECH_ANCHORS)
    if wc >= 30:
        return 1.0
    elif has_anchor:
        return max(0.75, min(1.0, wc / 30))
    else:
        return min(1.0, wc / 30)

def enrich_job_text(row):
    """
    Enrich job text for embedding.
    FIX 2: For sparse job descriptions (<15 words), fall back to candidate
    headline + summary from candidate_base (merged in earlier).
    Do NOT include company name — adds noise.
    """
    title       = safe_text(row.get('title'))
    description = safe_text(row.get('description'))
    industry    = safe_text(row.get('industry'))
    # Profile fields (present after merge with df_base)
    headline    = safe_text(row.get('headline'))
    summary     = safe_text(row.get('summary'))

    wc = len(description.split())

    if wc < 15:
        # Sparse job description — augment with candidate profile context
        # Cap summary at 300 chars so it doesn't swamp the job title
        profile_ctx = ' '.join(filter(None, [industry, headline, summary[:300]]))
        return f"{title}. {profile_ctx}. {description}".strip()
    else:
        return f"{title}. {description}".strip()

print("Functions defined: safe_text, description_confidence, enrich_job_text")


Functions defined: safe_text, description_confidence, enrich_job_text


In [13]:
# FIX 2: Merge candidate profile fields into df_jobs BEFORE enrichment ─────
# Secondary signal for sparse descriptions: headline, summary, current_title
PROFILE_COLS = [
    c for c in [
        'candidate_id', 'headline', 'summary', 'location',
        'country', 'years_of_experience', 'current_title',
    ]
    if c in df_base.columns
]
print(f"Profile columns available for merge: {PROFILE_COLS}")

df_jobs = df_jobs.merge(
    df_base[PROFILE_COLS].drop_duplicates('candidate_id'),
    on='candidate_id',
    how='left',
)
print(f"df_jobs shape after merge: {df_jobs.shape}")

# ── Apply text enrichment ─────────────────────────────────────────────────
df_jobs['enriched_text']          = df_jobs.apply(enrich_job_text, axis=1)
df_jobs['description_confidence'] = df_jobs['description'].apply(description_confidence)

print("\nConfidence distribution:")
print(df_jobs['description_confidence'].describe().round(3))
low_pct = (df_jobs['description_confidence'] < 0.75).mean() * 100
print(f"Low-confidence jobs (<0.75): {low_pct:.1f}%")

# ── Detect company column name ─────────────────────────────────────────────
company_col = next(
    (c for c in ['company','employer','organization','company_name'] if c in df_jobs.columns),
    None,
)
if company_col:
    print(f"\nCompany column found: '{company_col}'")
else:
    df_jobs['company'] = 'Unknown'
    company_col = 'company'
    print("\nNo company column found — defaulting to 'Unknown'")

# ── Fix date types ─────────────────────────────────────────────────────────
df_jobs['end_date']   = pd.to_datetime(df_jobs['end_date'],   errors='coerce')
df_jobs['start_date'] = pd.to_datetime(df_jobs['start_date'], errors='coerce')
df_jobs['is_current'] = df_jobs['is_current'].fillna(False).astype(bool)

# Fill NaT end_date for current roles
df_jobs.loc[df_jobs['is_current'], 'end_date'] = REFERENCE_DATE

# FIX 6: Always recompute duration_months after date cleanup.
# Do NOT reuse pre-existing values — current roles had stale end_dates before this cell.
df_jobs['duration_months'] = (
    (df_jobs['end_date'] - df_jobs['start_date']).dt.days / 30.44
).clip(lower=0)

df_jobs['months_since_end'] = (
    (REFERENCE_DATE - df_jobs['end_date']).dt.days / 30.44
).clip(lower=0)

# ── Save job_meta for ranking step ────────────────────────────────────────
META_COLS = [
    'candidate_id', 'title', company_col,
    'description_confidence', 'is_current',
    'end_date', 'duration_months', 'months_since_end',
]
df_meta_save = (
    df_jobs[META_COLS]
    .rename(columns={company_col: 'company'})
    .reset_index(drop=True)
)
df_meta_save.to_parquet(EMBED_DIR / 'job_meta.parquet', index=True)
print(f"\nSaved job_meta.parquet: {len(df_meta_save):,} rows → {EMBED_DIR / 'job_meta.parquet'}")


Profile columns available for merge: ['candidate_id', 'headline', 'summary', 'location', 'country', 'years_of_experience', 'current_title']
df_jobs shape after merge: (285346, 17)

Confidence distribution:
count    285346.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: description_confidence, dtype: float64
Low-confidence jobs (<0.75): 0.0%

Company column found: 'company'

Saved job_meta.parquet: 285,346 rows → artifacts\job_meta.parquet


In [14]:
from sentence_transformers import SentenceTransformer

JOB_EMB_PATH   = EMBED_DIR / 'job_embeddings.npy'
QUERY_EMB_PATH = EMBED_DIR / 'query_embeddings.npy'

print("Loading BAAI/bge-base-en-v1.5 ...")
model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)
print("Model loaded.\n")

# ── Encode job texts ───────────────────────────────────────────────────────
if JOB_EMB_PATH.exists():
    print(f"job_embeddings.npy already exists ({JOB_EMB_PATH.stat().st_size/1e6:.0f} MB).")
    print("Delete the file and re-run to re-embed.")
else:
    texts = df_jobs['enriched_text'].tolist()
    print(f"Encoding {len(texts):,} job texts on {device} (batch={BATCH_SIZE}) ...")
    t0 = time.time()
    job_embeddings = model.encode(
        texts,
        batch_size           = BATCH_SIZE,
        normalize_embeddings = True,   # L2-norm → dot product == cosine similarity
        show_progress_bar    = True,
        convert_to_numpy     = True,
    )
    elapsed = time.time() - t0
    print(f"Done in {elapsed/60:.1f} min — shape: {job_embeddings.shape}")
    np.save(JOB_EMB_PATH, job_embeddings)
    print(f"Saved → {JOB_EMB_PATH}  ({JOB_EMB_PATH.stat().st_size/1e6:.0f} MB)")

# ── Encode query strings ───────────────────────────────────────────────────
if QUERY_EMB_PATH.exists():
    print(f"\nquery_embeddings.npy already exists — skipping.")
else:
    print("\nEncoding 11 query strings ...")
    query_embeddings = model.encode(
        QUERY_TEXTS,
        batch_size           = 11,
        normalize_embeddings = True,
        show_progress_bar    = False,
        convert_to_numpy     = True,
    )
    np.save(QUERY_EMB_PATH, query_embeddings)
    print(f"Saved → {QUERY_EMB_PATH}  shape: {query_embeddings.shape}")


E:\Coding\Projects\redrob-ranker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading BAAI/bge-base-en-v1.5 ...


E:\Coding\Projects\redrob-ranker\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ajita\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8652.89it/s]


Model loaded.

Encoding 285,346 job texts on cpu (batch=128) ...


Batches:   1%|          | 21/2230 [02:37<4:35:31,  7.48s/it]


KeyboardInterrupt: 

---
## Part 3 — Ranking *(must complete in < 5 min in sandbox)*

Loads pre-computed `.npy` files only — no model calls at ranking time.


In [ ]:
job_embeddings   = np.load(EMBED_DIR / 'job_embeddings.npy')
query_embeddings = np.load(EMBED_DIR / 'query_embeddings.npy')
df_meta          = pd.read_parquet(EMBED_DIR / 'job_meta.parquet')

assert len(job_embeddings) == len(df_meta), (
    f"Row mismatch: embeddings={len(job_embeddings)}, meta={len(df_meta)}"
)
print(f"job_embeddings   : {job_embeddings.shape}")
print(f"query_embeddings : {query_embeddings.shape}")
print(f"job_meta rows    : {len(df_meta):,}")

# ── Matrix multiply: (N_jobs, 768) @ (768, 11) = (N_jobs, 11) ─────────────
t0 = time.time()
score_matrix = job_embeddings @ query_embeddings.T
print(f"\nScore matrix: {score_matrix.shape}  computed in {time.time()-t0:.2f}s")

score_matrix = np.clip(score_matrix, 0, None)   # clamp negatives (model noise)

for i, key in enumerate(QUERY_KEYS):
    df_meta[f'score_{key}'] = score_matrix[:, i]

print("\nRaw M-score distributions:")
print(df_meta[[f'score_{k}' for k in ['M1','M2','M3','M4']]].describe().round(3))


In [ ]:
# ── Must-have recency (steep) ─────────────────────────────────────────────
df_meta['recency_mh'] = np.select(
    condlist=[
        df_meta['is_current'],
        df_meta['months_since_end'] <= 18,
        df_meta['months_since_end'] <= 36,
        df_meta['months_since_end'] <= 60,
    ],
    choicelist=[1.25, 1.00, 0.65, 0.35],
    default=0.15,
)

# ── Nice-to-have recency (gentle) ────────────────────────────────────────
df_meta['recency_nth'] = np.select(
    condlist=[
        df_meta['is_current'],
        df_meta['months_since_end'] <= 36,
        df_meta['months_since_end'] <= 60,
    ],
    choicelist=[1.15, 1.00, 0.75],
    default=0.50,
)

print("Recency weight distributions:")
print(df_meta[['recency_mh','recency_nth']].describe().round(3))
print(f"\nCurrent-job rows: {df_meta['is_current'].sum():,}")


In [ ]:
# ── Per-job adjusted M scores ─────────────────────────────────────────────
M_COLS = ['M1','M2','M3','M4']
for m in M_COLS:
    df_meta[f'adj_{m}'] = (
        df_meta[f'score_{m}']
        * df_meta['description_confidence']
        * df_meta['recency_mh']
    ).clip(0, 1.0)

# FIX 5: Seed df_cand from ALL survivor_ids, not just those with job rows.
# Candidates with no jobs get M scores of 0 (will naturally fall outside top-100).
df_cand = pd.DataFrame(
    index=pd.Index(sorted(survivor_ids), name='candidate_id')
)

recency_mh_sum = df_meta.groupby('candidate_id')['recency_mh'].sum()

# ── Per-candidate per-M: 70/30 blend of best-job + recency-weighted avg ────
for m in M_COLS:
    best  = df_meta.groupby('candidate_id')[f'adj_{m}'].max()
    w_avg = (
        df_meta.groupby('candidate_id')[f'adj_{m}'].sum() / recency_mh_sum
    ).fillna(0)
    df_cand[f'{m}_score'] = (
        (0.70 * best + 0.30 * w_avg)
        .reindex(df_cand.index)   # aligns to all survivors, NaN for no-job candidates
        .fillna(0)
        .clip(0, 1.0)
    )

print("Per-candidate M-score distributions:")
print(df_cand[['M1_score','M2_score','M3_score','M4_score']].describe().round(3))
print(f"\nZero-score M1 candidates: {(df_cand['M1_score'] == 0).sum():,}")


In [ ]:
# ── Geometric mean of M1–M4 → must_have_score ─────────────────────────────
m_arr = df_cand[['M1_score','M2_score','M3_score','M4_score']].values.clip(GMEAN_FLOOR, 1.0)
df_cand['must_have_score'] = scipy_gmean(m_arr, axis=1)

print("must_have_score distribution:")
print(df_cand['must_have_score'].describe().round(3))
q = df_cand['must_have_score'].quantile([0.50, 0.75, 0.90, 0.95, 0.99])
print("\nKey quantiles:")
print(q.round(3).to_string())
if df_cand['must_have_score'].median() > 0.5:
    print("\n⚠️  WARNING: median > 0.5 — scoring is too generous")
else:
    print("\n✅  Right-skewed distribution as expected")


In [ ]:
N_COLS = ['N1','N2','N3','N4']
for n in N_COLS:
    df_meta[f'adj_{n}'] = (
        df_meta[f'score_{n}']
        * df_meta['description_confidence']
        * df_meta['recency_nth']
    ).clip(0, 1.0)

df_meta['best_nth'] = df_meta[[f'adj_{n}' for n in N_COLS]].max(axis=1)

recency_nth_sum = df_meta.groupby('candidate_id')['recency_nth'].sum()
df_cand['nth_raw'] = (
    df_meta.groupby('candidate_id')['best_nth'].sum() / recency_nth_sum
).reindex(df_cand.index).fillna(0).clip(0, 1.0)

df_cand['nth_bonus'] = (df_cand['nth_raw'] * 0.20).clip(0, 0.20)

print("Nice-to-have bonus distribution:")
print(df_cand['nth_bonus'].describe().round(3))


In [ ]:
# ── READ THESE before touching threshold constants in the next cell ────────
print("IR score quantiles (candidate-level max):")
cand_ir = df_meta.groupby('candidate_id')['score_IR'].max().reindex(df_cand.index).fillna(0)
print(cand_ir.quantile([0.25, 0.50, 0.75, 0.90]).round(3).to_dict())

print("\nCV score quantiles (candidate-level max):")
cand_cv = df_meta.groupby('candidate_id')['score_CV'].max().reindex(df_cand.index).fillna(0)
print(cand_cv.quantile([0.25, 0.50, 0.75, 0.90]).round(3).to_dict())

print("\nHANDS_ON score quantiles (candidate-level max):")
cand_ho_all = df_meta.groupby('candidate_id')['score_HANDS_ON'].max().reindex(df_cand.index).fillna(0)
print(cand_ho_all.quantile([0.25, 0.50, 0.75, 0.90]).round(3).to_dict())

print("\n→ Adjust threshold constants in the next cell based on these distributions.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CALIBRATE these thresholds after reviewing the quantile output above.
# ══════════════════════════════════════════════════════════════════════════

# ── Domain multiplier ──────────────────────────────────────────────────────
CV_HIGH_THRESH = 0.65
CV_MED_THRESH  = 0.50
IR_LOW_THRESH  = 0.40
IR_MED_THRESH  = 0.50

def domain_mult_fn(ir, cv):
    if cv > CV_HIGH_THRESH and ir < IR_LOW_THRESH:
        return 0.65
    elif cv > CV_MED_THRESH and ir < IR_MED_THRESH:
        return 0.90
    return 1.00

df_cand['domain_multiplier'] = [
    domain_mult_fn(cand_ir.get(cid, 0), cand_cv.get(cid, 0))
    for cid in df_cand.index
]
print("Domain multiplier distribution:")
print(df_cand['domain_multiplier'].value_counts().sort_index())

# ── Hands-on multiplier ────────────────────────────────────────────────────
recent_mask = df_meta['is_current'] | (df_meta['months_since_end'] <= 18)
recent_ho   = (
    df_meta[recent_mask]
    .groupby('candidate_id')['score_HANDS_ON']
    .max()
    .reindex(df_cand.index)
    .fillna(0)
)

HO_THRESHOLDS = [(0.75, 1.00), (0.55, 0.85), (0.35, 0.70)]

def hands_on_mult_fn(score):
    for thresh, mult in HO_THRESHOLDS:
        if score >= thresh:
            return mult
    return 0.55

df_cand['hands_on_multiplier'] = recent_ho.map(hands_on_mult_fn)
print("\nHands-on multiplier distribution:")
print(df_cand['hands_on_multiplier'].value_counts().sort_index())

# ── Job-hopper multiplier (deterministic, no embedding) ───────────────────
completed = df_meta[~df_meta['is_current']].copy()

median_tenure = completed.groupby('candidate_id')['duration_months'].median()
last3_avg = (
    completed.sort_values('months_since_end')
             .groupby('candidate_id')
             .head(3)
             .groupby('candidate_id')['duration_months']
             .mean()
)
current_dur = (
    df_meta[df_meta['is_current']]
    .groupby('candidate_id')['duration_months']
    .max()
)

def hopper_mult_fn(med, last3, cur):
    if   med >= 24 and last3 >= 18:  mult = 1.00
    elif med >= 18 or  last3 >= 18:  mult = 0.92
    elif med >= 12:                   mult = 0.82
    else:                             mult = 0.70
    if cur < 6:
        mult *= 0.95
    return mult

df_cand['job_hopper_multiplier'] = [
    hopper_mult_fn(
        median_tenure.get(cid, 24),
        last3_avg.get(cid, 24),
        current_dur.get(cid, 24),
    )
    for cid in df_cand.index
]
print("\nJob-hopper multiplier distribution:")
print(df_cand['job_hopper_multiplier'].value_counts().sort_index())


In [ ]:
# ── Availability ──────────────────────────────────────────────────────────
avail = df_avail.set_index('candidate_id')['availability_multiplier']
df_cand['availability_multiplier'] = df_cand.index.map(avail).fillna(1.0)

# FIX 4: Trap multiplier — require explicit column, no boolean auto-detection ─
TRAP_PATH = DATA_DIR / 'trap_scores.parquet'
if TRAP_PATH.exists():
    df_trap = pd.read_parquet(TRAP_PATH).set_index('candidate_id')

    if 'trap_multiplier' not in df_trap.columns:
        raise ValueError(
            "trap_scores.parquet exists but has no 'trap_multiplier' column.\n"
            f"Columns found: {list(df_trap.columns)}\n"
            "Rename the correct column to 'trap_multiplier' (float, 0.0–1.0) before continuing."
        )

    df_cand['trap_multiplier'] = (
        df_cand.index.map(df_trap['trap_multiplier']).fillna(1.0)
    )
    print(f"Trap multiplier loaded. Distribution:")
    print(df_cand['trap_multiplier'].describe().round(3))
else:
    df_cand['trap_multiplier'] = 1.0
    print("Step 4 artifact not found — trap_multiplier = 1.0  (run Step 4 before final submission)")

# ── Final score assembly (multiplicative) ─────────────────────────────────
df_cand['semantic_score'] = (
    df_cand['must_have_score']
    * (1 + df_cand['nth_bonus'])
    * df_cand['domain_multiplier']
    * df_cand['hands_on_multiplier']
)

df_cand['final_score_raw'] = (
    df_cand['semantic_score']
    * df_cand['availability_multiplier']
    * df_cand['job_hopper_multiplier']
    * df_cand['trap_multiplier']
)

# ALWAYS rank on raw — floor is ONLY for the display column
df_cand['final_score'] = df_cand['final_score_raw'].clip(lower=0.10)

print("\nFinal score (raw) distribution:")
print(df_cand['final_score_raw'].describe().round(4))


## Part 4 — Output

In [ ]:
# FIX 7: Best matching job uses geometric mean of adj M scores, not single-M max.
# A job that only matches M1 should not be chosen over a balanced one.
df_meta['composite_M'] = scipy_gmean(
    df_meta[[f'adj_{m}' for m in M_COLS]].clip(GMEAN_FLOOR, 1.0),
    axis=1,
)
best_job = (
    df_meta.sort_values('composite_M', ascending=False)
           .groupby('candidate_id')
           .first()
           [['title','company','description_confidence']]
           .rename(columns={
               'title':                  'best_matching_job_title',
               'company':                'best_matching_job_company',
               'description_confidence': 'best_job_confidence',
           })
)
df_cand = df_cand.join(best_job)
df_cand['best_matching_job_title']   = df_cand['best_matching_job_title'].fillna('Unknown')
df_cand['best_matching_job_company'] = df_cand['best_matching_job_company'].fillna('Unknown')
df_cand['best_job_confidence']       = df_cand['best_job_confidence'].fillna(0.0)

# ── Confidence flags ───────────────────────────────────────────────────────
df_cand['low_description_confidence'] = df_cand['best_job_confidence'] < 0.75
all_sparse = df_meta.groupby('candidate_id')['description_confidence'].max().lt(0.75)
df_cand['all_sparse'] = df_cand.index.map(all_sparse).fillna(True)

# ── Reasoning string builder ───────────────────────────────────────────────
def build_reason(row):
    strengths = []
    if row['M1_score'] > 0.60: strengths.append('retrieval/search systems')
    if row['M2_score'] > 0.60: strengths.append('vector DB / hybrid search')
    if row['M3_score'] > 0.60: strengths.append('ranking evaluation frameworks')
    if row['M4_score'] > 0.60: strengths.append('hands-on production coding')

    flags = []
    if row['job_hopper_multiplier']   < 0.85: flags.append('job stability concern')
    if row['domain_multiplier']       < 1.00: flags.append('partial domain mismatch')
    if row['low_description_confidence']:     flags.append('limited description detail')
    if row['availability_multiplier'] < 0.80: flags.append('availability friction')
    if row.get('all_sparse', False):          flags.append('all jobs have sparse descriptions')

    strength_str = ', '.join(strengths) if strengths else 'general ML background'
    flag_str     = ('. Note: ' + '; '.join(flags)) if flags else ''
    return (
        f"Evidence of {strength_str} from "
        f"{row['best_matching_job_title']} at {row['best_matching_job_company']} "
        f"(M1:{row['M1_score']:.2f} M2:{row['M2_score']:.2f} "
        f"M3:{row['M3_score']:.2f} M4:{row['M4_score']:.2f}){flag_str}."
    )

# ── Top-100 ────────────────────────────────────────────────────────────────
top100 = df_cand.nlargest(100, 'final_score_raw').copy()
top100['semantic_reason'] = top100.apply(build_reason, axis=1)

OUTPUT_COLS = [
    'final_score_raw', 'final_score', 'semantic_score', 'must_have_score',
    'M1_score', 'M2_score', 'M3_score', 'M4_score', 'nth_bonus',
    'domain_multiplier', 'hands_on_multiplier', 'job_hopper_multiplier',
    'availability_multiplier', 'trap_multiplier',
    'best_matching_job_title', 'best_matching_job_company',
    'low_description_confidence', 'all_sparse', 'semantic_reason',
]
top100 = top100[OUTPUT_COLS].reset_index()

OUT_PATH = OUTPUT_DIR / 'top100_ranked.csv'
top100.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")
print(f"\nTop 10 snapshot:")
print(
    top100[['candidate_id','final_score_raw','must_have_score','best_matching_job_title']]
    .head(10)
    .to_string(index=False)
)


## Part 5 — Validation *(run before submitting)*

In [ ]:
print("=" * 60)
print("VALIDATION CHECKS")
print("=" * 60)

# [1] Score distribution
print("\n[1] must_have_score distribution:")
q = df_cand['must_have_score'].quantile([0.50, 0.75, 0.90, 0.95, 0.99])
print(q.round(3).to_string())
if df_cand['must_have_score'].median() > 0.50:
    print("⚠️  WARNING: Median > 0.5 — scoring too generous")
else:
    print("✅  Median < 0.5 — right-skewed as expected")

# [2] M1/M2 correlation
print("\n[2] M1/M2 Pearson correlation:")
corr = df_cand[['M1_score','M2_score']].corr().iloc[0,1]
print(f"    r = {corr:.3f}")
if corr > 0.90:
    print("⚠️  WARNING: >0.90 — M1/M2 redundant, consider averaging before gmean")
elif corr < 0.50:
    print("⚠️  WARNING: <0.50 — unexpectedly low, check query strings")
else:
    print("✅  In expected 0.6–0.8 range")

# [3] Domain distributions
print("\n[3] Domain score quantiles:")
print("IR:", cand_ir.quantile([0.25,.50,.75,.90]).round(3).to_dict())
print("CV:", cand_cv.quantile([0.25,.50,.75,.90]).round(3).to_dict())

# [4] Data-engineer sanity test (manual)
print("\n[4] Data engineer test — find a Kafka/Airflow candidate and verify:")
print("    Expected: M1~0.10, M2~0.05, M3~0.05  (manual spot-check)")

# [5] Floor check
floored = (df_cand['final_score'] == 0.10).sum()
print(f"\n[5] Candidates floored at 0.10: {floored:,} ({floored/len(df_cand)*100:.1f}%)")
if floored > len(df_cand) * 0.10:
    print("⚠️  WARNING: >10% floored — check if multipliers are too aggressive")
else:
    print("✅  Floor rate looks fine")

# [6] Honeypot check
if (top100['trap_multiplier'] < 1.0).any():
    trap_count = (top100['trap_multiplier'] < 1.0).sum()
    print(f"\n[6] Trap candidates in top 100: {trap_count}")
    if trap_count >= 10:
        print("⚠️  WARNING: ≥10 honeypots in top 100 — SUBMISSION WILL BE DISQUALIFIED")
    else:
        print("✅  Honeypot count OK")
else:
    print("\n[6] Trap multiplier not applied (Step 4 not run) — do before final submission")

# [7] Top-20 spot-check
print("\n[7] Top-20 spot-check:")
print(
    top100[['candidate_id','final_score_raw','M1_score','M2_score',
            'M3_score','M4_score','best_matching_job_title']]
    .head(20)
    .to_string(index=False)
)
print("\n✅  Validation complete. Fix any warnings above before submitting.")
